# Funciones · Excepciones

Capturar y lanzar excepciones, crear excepciones propias, validar entradas y agrupar errores.

## 1. División segura

Escribir una función `divide` que devuelva el cociente de dos valores, `None` si el divisor es cero y un mensaje si los operandos no son numéricos.

In [1]:
def divide(a, b):
    try:
        return a / b
    except ZeroDivisionError:
        return None
    except TypeError:
        return "operandos no numéricos"

print(divide(10, 4), divide(1, 0), divide("1", 2))
assert divide(10, 4) == 2.5
assert divide(1, 0) is None

2.5 None operandos no numéricos


## 2. Convertir cadenas a enteros

Escribir una función que reciba una lista de cadenas y devuelva dos listas: los enteros que se han podido convertir y las cadenas que no.

In [2]:
def a_enteros(cadenas):
    validos, invalidos = [], []
    for cadena in cadenas:
        try:
            validos.append(int(cadena))
        except ValueError:
            invalidos.append(cadena)
    return validos, invalidos

validos, invalidos = a_enteros(["12", "7", "tres", "4.5", " 9 ", ""])
print(validos, invalidos)
assert validos == [12, 7, 9]
assert invalidos == ["tres", "4.5", ""]

[12, 7, 9] ['tres', '4.5', '']


## 3. `try` / `except` / `else` / `finally`

Escribir una función que registre qué bloques se ejecutan al convertir un texto a entero. Después, escribir otra que calcule la media de las notas de un fichero ignorando las líneas no válidas y cerrando siempre el fichero.

In [3]:
def procesa(texto):
    pasos = []
    try:
        numero = int(texto)
    except ValueError:
        pasos.append("except")
    else:
        pasos.append(f"else ({numero * 2})")
    finally:
        pasos.append("finally")
    return pasos

print(procesa("21"))
print(procesa("abc"))
assert procesa("21") == ["else (42)", "finally"]
assert procesa("abc") == ["except", "finally"]

['else (42)', 'finally']
['except', 'finally']


In [4]:
with open("notas.txt", "w", encoding="utf-8") as f:
    f.write("7\n8.5\nnueve\n6\n")

def media_fichero(ruta):
    f = None
    try:
        f = open(ruta, encoding="utf-8")
    except FileNotFoundError:
        print(f"no existe {ruta}")
        return None
    else:
        notas = []
        for linea in f:
            try:
                notas.append(float(linea))
            except ValueError:
                print(f"línea ignorada: {linea.strip()!r}")
        return sum(notas) / len(notas)
    finally:
        # se ejecuta incluso tras el return
        if f:
            f.close()
            print("fichero cerrado")

media = media_fichero("notas.txt")
print("media:", media)
print(media_fichero("no-existe.txt"))
assert round(media, 4) == 7.1667

línea ignorada: 'nueve'
fichero cerrado
media: 7.166666666666667
no existe no-existe.txt
None


## 4. Lanzar excepciones con `raise`

Escribir una función `valida_edad` que lance `TypeError` si la edad no es un entero y `ValueError` si no está entre 0 y 130.

In [5]:
def valida_edad(edad):
    if not isinstance(edad, int):
        raise TypeError(f"la edad debe ser un entero, no {type(edad).__name__}")
    if not 0 <= edad <= 130:
        raise ValueError(f"edad fuera de rango: {edad}")
    return edad

for valor in (25, -3, "veinte", 200):
    try:
        print(f"{valor!r} → {valida_edad(valor)}")
    except (TypeError, ValueError) as e:
        print(f"{valor!r} → {type(e).__name__}: {e}")

25 → 25
-3 → ValueError: edad fuera de rango: -3
'veinte' → TypeError: la edad debe ser un entero, no str
200 → ValueError: edad fuera de rango: 200


## 5. Jerarquía de excepciones propias

Crear una excepción base `ErrorBancario` y las subclases `SaldoInsuficiente` (que guarde cuánto falta), `CuentaBloqueada` e `ImporteInvalido` (que además sea un `ValueError`). Crear una clase `Cuenta` con un método `retirar` que las lance.

In [6]:
class ErrorBancario(Exception):
    """Base de los errores del banco."""

class SaldoInsuficiente(ErrorBancario):
    def __init__(self, saldo, importe):
        super().__init__(f"saldo {saldo} € insuficiente para retirar {importe} €")
        self.falta = importe - saldo

class CuentaBloqueada(ErrorBancario):
    pass

class ImporteInvalido(ErrorBancario, ValueError):
    pass

class Cuenta:
    def __init__(self, titular, saldo=0):
        self.titular = titular
        self.saldo = saldo
        self.bloqueada = False

    def retirar(self, importe):
        if importe <= 0:
            raise ImporteInvalido(f"importe no válido: {importe}")
        if self.bloqueada:
            raise CuentaBloqueada(f"la cuenta de {self.titular} está bloqueada")
        if importe > self.saldo:
            raise SaldoInsuficiente(self.saldo, importe)
        self.saldo -= importe
        return self.saldo

In [7]:
cuenta = Cuenta("Ana", 100)

for importe in (30, 120, -5):
    try:
        print(f"retira {importe} → saldo {cuenta.retirar(importe)}")
    except SaldoInsuficiente as e:
        print(f"retira {importe} → {e} (faltan {e.falta} €)")
    except ErrorBancario as e:
        print(f"retira {importe} → {type(e).__name__}: {e}")

cuenta.bloqueada = True
try:
    cuenta.retirar(10)
except ErrorBancario as e:
    print(type(e).__name__, "→", e)

assert cuenta.saldo == 70
assert issubclass(ImporteInvalido, ValueError)

retira 30 → saldo 70
retira 120 → saldo 70 € insuficiente para retirar 120 € (faltan 50 €)
retira -5 → ImporteInvalido: importe no válido: -5
CuentaBloqueada → la cuenta de Ana está bloqueada


## 6. Re-lanzar con `raise ... from`

Escribir una función que lea líneas `clave = valor` con valores enteros y, si alguno no es numérico, lance `ErrorConfiguracion` encadenando el `ValueError` original. Escribir también una búsqueda que oculte el `KeyError` original con `from None`.

In [8]:
class ErrorConfiguracion(Exception):
    pass

def lee_config(lineas):
    config = {}
    for n, linea in enumerate(lineas, 1):
        clave, _, valor = linea.partition("=")
        try:
            config[clave.strip()] = int(valor)
        except ValueError as e:
            raise ErrorConfiguracion(f"línea {n}: valor no numérico en {clave.strip()!r}") from e
    return config

print(lee_config(["puerto = 8080", "hilos=4"]))

try:
    lee_config(["puerto = 8080", "hilos = cuatro"])
except ErrorConfiguracion as e:
    print("Error:", e)
    print("Causa:", repr(e.__cause__))

{'puerto': 8080, 'hilos': 4}
Error: línea 2: valor no numérico en 'hilos'
Causa: ValueError("invalid literal for int() with base 10: ' cuatro'")


In [9]:
precios = {"pan": 1.2, "leche": 0.95}

def busca_precio(producto):
    try:
        return precios[producto]
    except KeyError:
        raise LookupError(f"producto desconocido: {producto}") from None

try:
    busca_precio("queso")
except LookupError as e:
    print(e, "| causa:", e.__cause__, "| contexto oculto:", e.__suppress_context__)

producto desconocido: queso | causa: None | contexto oculto: True


## 7. Validar un formulario

Escribir una función que valide un registro con `nombre` (mínimo 2 caracteres), `email` y `edad` (entero, mayor de edad). Si hay errores, lanzar una excepción `ErrorValidacion` que los incluya todos; si no, devolver los datos normalizados.

In [10]:
import re

class ErrorValidacion(Exception):
    def __init__(self, errores):
        super().__init__(f"{len(errores)} campo(s) no válido(s)")
        self.errores = errores

def valida_registro(datos):
    errores = {}
    nombre = datos.get("nombre", "").strip()
    if len(nombre) < 2:
        errores["nombre"] = "mínimo 2 caracteres"
    email = datos.get("email", "").strip()
    if not re.fullmatch(r"[\w.+-]+@[\w-]+\.[\w.]+", email):
        errores["email"] = "formato no válido"
    try:
        edad = int(datos.get("edad", ""))
    except ValueError:
        errores["edad"] = "debe ser un número entero"
    else:
        if edad < 18:
            errores["edad"] = "debe ser mayor de edad"
    if errores:
        raise ErrorValidacion(errores)
    return {"nombre": nombre.title(), "email": email.lower(), "edad": edad}

print(valida_registro({"nombre": " carmen ", "email": "Carmen@Correo.es", "edad": "34"}))

for datos in ({"nombre": "J", "email": "jose@", "edad": "dieciséis"}, {"nombre": "Luis", "email": "luis@correo.es", "edad": "15"}):
    try:
        valida_registro(datos)
    except ErrorValidacion as e:
        print(e, e.errores)

{'nombre': 'Carmen', 'email': 'carmen@correo.es', 'edad': 34}
3 campo(s) no válido(s) {'nombre': 'mínimo 2 caracteres', 'email': 'formato no válido', 'edad': 'debe ser un número entero'}
1 campo(s) no válido(s) {'edad': 'debe ser mayor de edad'}


## 8. `assert` frente a excepciones

Escribir una función `media` que lance `ValueError` si la lista está vacía (error del usuario) y use `assert` para comprobar un invariante interno (error del programador). Comprobar que los `assert` desaparecen al ejecutar Python con `-O`.

In [11]:
def media(numeros):
    if not numeros:
        raise ValueError("la lista no puede estar vacía")
    resultado = sum(numeros) / len(numeros)
    assert min(numeros) <= resultado <= max(numeros), "la media debe estar entre el mínimo y el máximo"
    return resultado

print(media([4, 8, 6]))
try:
    media([])
except ValueError as e:
    print("ValueError:", e)

6.0
ValueError: la lista no puede estar vacía


In [12]:
import os
import subprocess
import sys

programa = "assert 1 > 2, 'falla el assert'\nprint('assert ignorado')"
sin_colores = {**os.environ, "PYTHON_COLORS": "0"}

for opciones in ([], ["-O"]):
    salida = subprocess.run([sys.executable, *opciones, "-c", programa], capture_output=True, text=True, env=sin_colores)
    ultima_linea = (salida.stdout or salida.stderr).strip().splitlines()[-1]
    print(f"python {' '.join(opciones) or '  '} → {ultima_linea}")

python    → AssertionError: falla el assert
python -O → assert ignorado


## 9. Reintentos y re-lanzar

Escribir una función `reintenta` que llame a otra función hasta `intentos` veces si lanza `ConnectionError` y, si fallan todos los intentos, vuelva a lanzar la última excepción con `raise`. Probarla con un servicio simulado que falla las primeras llamadas.

In [13]:
def reintenta(funcion, intentos=3, excepciones=(ConnectionError,)):
    for intento in range(1, intentos + 1):
        try:
            return funcion()
        except excepciones as e:
            print(f"  intento {intento} fallido: {e}")
            if intento == intentos:
                raise

def servicio_inestable(fallos):
    llamadas = 0

    def llamar():
        nonlocal llamadas
        llamadas += 1
        if llamadas <= fallos:
            raise ConnectionError("tiempo de espera agotado")
        return "datos recibidos"

    return llamar

print(reintenta(servicio_inestable(2)))

try:
    reintenta(servicio_inestable(5))
except ConnectionError as e:
    print("sin éxito:", e)

  intento 1 fallido: tiempo de espera agotado
  intento 2 fallido: tiempo de espera agotado
datos recibidos
  intento 1 fallido: tiempo de espera agotado
  intento 2 fallido: tiempo de espera agotado
  intento 3 fallido: tiempo de espera agotado
sin éxito: tiempo de espera agotado


## 10. `ExceptionGroup` y `except*`

Escribir una función que valide una lista de pedidos y, en lugar de detenerse en el primer error, lance un `ExceptionGroup` con todos. Manejar por separado los `ValueError` y los `TypeError` con `except*`.

In [14]:
def valida_pedido(pedido):
    cantidad = pedido["cantidad"]
    if not isinstance(cantidad, int):
        raise TypeError(f"pedido {pedido['id']}: la cantidad debe ser entera")
    if cantidad <= 0:
        raise ValueError(f"pedido {pedido['id']}: la cantidad debe ser positiva")

def valida_pedidos(pedidos):
    errores = []
    for pedido in pedidos:
        try:
            valida_pedido(pedido)
        except (TypeError, ValueError) as e:
            errores.append(e)
    if errores:
        raise ExceptionGroup("pedidos no válidos", errores)

pedidos = [
    {"id": 1, "cantidad": 3},
    {"id": 2, "cantidad": -1},
    {"id": 3, "cantidad": "dos"},
    {"id": 4, "cantidad": 0},
]

try:
    valida_pedidos(pedidos)
except* ValueError as grupo:
    for e in grupo.exceptions:
        print("Valor:", e)
except* TypeError as grupo:
    for e in grupo.exceptions:
        print("Tipo: ", e)

Valor: pedido 2: la cantidad debe ser positiva
Valor: pedido 4: la cantidad debe ser positiva
Tipo:  pedido 3: la cantidad debe ser entera
